# Hyundai Cars: Maintenance Type Prediction with ClearML

This notebook fixes the target column issue by using `maintenance_type` as the label.
It compares **Random Forest** and **SVM** with **Optuna HPO**, tracks the run in **ClearML**, and saves the best trained pipeline bundle.
            


In [ ]:
from pathlib import Path
import os
import pickle

import numpy as np
import optuna
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from clearml import Logger, Task
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, RobustScaler
from sklearn.svm import SVC

optuna.logging.set_verbosity(optuna.logging.WARNING)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

RANDOM_STATE = 42
TEST_SIZE = 0.2
HPO_TRIALS_PER_MODEL = 10
CV_FOLDS = 3
CLEARML_PROJECT = "605-Vehicle_Maintainance-project"
CLEARML_TASK_NAME = "cars_hyundai notebook - maintenance_type svm vs rf"
            


In [ ]:
data_path = Path("..") / "data" / "cars_hyundai.csv"
print(f"Dataset path: {data_path.resolve()}")

df_raw = pd.read_csv(data_path)
print(f"Raw shape: {df_raw.shape}")
df_raw.head()
            


In [ ]:
rename_map = {
    "Engine Temperature (?C)": "engine_temperature_c",
    "Brake Pad Thickness (mm)": "brake_pad_thickness_mm",
    "Tire Pressure (PSI)": "tire_pressure_psi",
    "Maintenance Type": "maintenance_type",
    "Anomaly Indication": "anomaly_indication",
}

df = df_raw.rename(columns=rename_map).drop_duplicates().reset_index(drop=True)

TARGET_COLUMN = "maintenance_type"
NUMERIC_FEATURES = [
    "engine_temperature_c",
    "brake_pad_thickness_mm",
    "tire_pressure_psi",
    "anomaly_indication",
]

print(df.columns.tolist())
df.head()
            


In [ ]:
print("Missing values:")
display(df.isna().sum().to_frame("missing_count"))

print("Target distribution:")
display(df[TARGET_COLUMN].value_counts().to_frame("count"))

print("Data types:")
display(df.dtypes.to_frame("dtype"))
            


In [ ]:
# ── Target-class distribution ─────────────────────────────────────────────────
target_distribution = df[TARGET_COLUMN].value_counts().sort_index()

# Reindex to guarantee all expected classes appear even if one has 0 samples.
# This prevents the "pie only shows 2 labels" bug when a class is absent.
all_classes = sorted(df[TARGET_COLUMN].unique())
target_distribution = target_distribution.reindex(all_classes, fill_value=0)

pie_labels = [str(cls) for cls in target_distribution.index]
palette = sns.color_palette("Set2", n_colors=len(target_distribution))

fig, axes = plt.subplots(1, 3, figsize=(17, 4))

# --- Bar chart ---
sns.countplot(data=df, x=TARGET_COLUMN, palette="Set2", order=all_classes, ax=axes[0])
axes[0].set_title("Maintenance Type Distribution")
axes[0].set_xlabel("Maintenance Type")
axes[0].set_ylabel("Count")
axes[0].tick_params(axis="x", rotation=15)

# --- Pie chart (safe: reindexed so all classes always have a slice) ---
target_distribution.plot(
    kind="pie",
    autopct="%.1f%%",
    ax=axes[1],
    labels=pie_labels,
    colors=palette,
    pctdistance=0.80,
    startangle=90,
)
axes[1].set_ylabel("")
axes[1].set_title("Target Share")

# --- Spearman correlation heat-map ---
corr = df[NUMERIC_FEATURES].corr(method="spearman")
sns.heatmap(corr, annot=True, cmap="coolwarm", fmt=".2f", ax=axes[2])
axes[2].set_title("Spearman Correlation (features)")

plt.tight_layout()
plt.show()


In [ ]:
X = df[NUMERIC_FEATURES].copy()
y_raw = df[TARGET_COLUMN].copy()

label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y_raw)
class_names = label_encoder.classes_.tolist()
label_mapping = dict(enumerate(class_names))

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)

print("Classes:", label_mapping)
print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)
            


In [ ]:
if not os.getenv("CLEARML_API_ACCESS_KEY") and not os.getenv("CLEARML_OFFLINE_MODE"):
    os.environ["CLEARML_OFFLINE_MODE"] = "1"
    print("ClearML credentials not detected; running in offline mode.")

task = Task.init(
    project_name=CLEARML_PROJECT,
    task_name=CLEARML_TASK_NAME,
    task_type=Task.TaskTypes.training,
    reuse_last_task_id=False,
)
logger = Logger.current_logger()

task.connect(
    {
        "target_column": TARGET_COLUMN,
        "numeric_features": NUMERIC_FEATURES,
        "random_state": RANDOM_STATE,
        "test_size": TEST_SIZE,
        "hpo_trials_per_model": HPO_TRIALS_PER_MODEL,
        "cv_folds": CV_FOLDS,
        "classes": class_names,
    }
)
            


In [ ]:
preprocessor = ColumnTransformer(
    transformers=[("num", RobustScaler(), NUMERIC_FEATURES)],
    remainder="drop",
)
cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)


def build_rf(params):
    return Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            (
                "model",
                RandomForestClassifier(
                    n_estimators=int(params["n_estimators"]),
                    max_depth=int(params["max_depth"]),
                    min_samples_leaf=int(params["min_samples_leaf"]),
                    min_samples_split=int(params["min_samples_split"]),
                    max_features=params["max_features"],
                    random_state=RANDOM_STATE,
                    n_jobs=-1,
                ),
            ),
        ]
    )


def build_svm(params):
    return Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            (
                "model",
                SVC(
                    C=float(params["C"]),
                    kernel=params["kernel"],
                    gamma=params["gamma"],
                    degree=int(params.get("degree", 3)),
                    probability=True,
                    random_state=RANDOM_STATE,
                ),
            ),
        ]
    )


def tune_random_forest():
    def objective(trial):
        params = {
            "n_estimators": trial.suggest_int("n_estimators", 100, 500),
            "max_depth": trial.suggest_int("max_depth", 3, 20),
            "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 8),
            "min_samples_split": trial.suggest_int("min_samples_split", 2, 12),
            "max_features": trial.suggest_categorical("max_features", ["sqrt", "log2"]),
        }
        pipeline = build_rf(params)
        score = float(cross_val_score(pipeline, X_train, y_train, cv=cv, scoring="f1_weighted", n_jobs=-1).mean())
        logger.report_scalar("hpo/cv_f1", "Random Forest", score, trial.number)
        return score

    study = optuna.create_study(direction="maximize", study_name="hyundai_rf_hpo")
    study.optimize(objective, n_trials=HPO_TRIALS_PER_MODEL, show_progress_bar=False)
    logger.report_scalar("hpo/model_best_cv_f1", "Random Forest", study.best_value, 0)
    return {
        "model_name": "Random Forest",
        "best_params": study.best_params,
        "cv_f1_weighted": float(study.best_value),
        "pipeline": build_rf(study.best_params),
    }


def tune_svm():
    def objective(trial):
        kernel = trial.suggest_categorical("kernel", ["linear", "rbf", "poly"])
        params = {
            "C": trial.suggest_float("C", 0.1, 50.0, log=True),
            "kernel": kernel,
            "gamma": trial.suggest_categorical("gamma", ["scale", "auto"]),
            "degree": trial.suggest_int("degree", 2, 4) if kernel == "poly" else 3,
        }
        pipeline = build_svm(params)
        score = float(cross_val_score(pipeline, X_train, y_train, cv=cv, scoring="f1_weighted", n_jobs=-1).mean())
        logger.report_scalar("hpo/cv_f1", "SVM", score, trial.number)
        return score

    study = optuna.create_study(direction="maximize", study_name="hyundai_svm_hpo")
    study.optimize(objective, n_trials=HPO_TRIALS_PER_MODEL, show_progress_bar=False)
    logger.report_scalar("hpo/model_best_cv_f1", "SVM", study.best_value, 0)
    return {
        "model_name": "SVM",
        "best_params": study.best_params,
        "cv_f1_weighted": float(study.best_value),
        "pipeline": build_svm(study.best_params),
    }
            


In [ ]:
rf_result = tune_random_forest()
svm_result = tune_svm()

model_results = []
for result in [rf_result, svm_result]:
    pipeline = result["pipeline"]
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)

    model_results.append(
        {
            "model": result["model_name"],
            "cv_f1_weighted": result["cv_f1_weighted"],
            "test_f1_weighted": float(f1_score(y_test, y_pred, average="weighted")),
            "test_precision_weighted": float(precision_score(y_test, y_pred, average="weighted", zero_division=0)),
            "test_recall_weighted": float(recall_score(y_test, y_pred, average="weighted", zero_division=0)),
            "test_accuracy": float(accuracy_score(y_test, y_pred)),
            "best_params": result["best_params"],
            "pipeline": pipeline,
        }
    )

results_df = pd.DataFrame(model_results).sort_values(by=["cv_f1_weighted", "test_f1_weighted"], ascending=False)
display(results_df.drop(columns=["pipeline"]))
logger.report_table("hyundai", "model_comparison", iteration=0, table_plot=results_df.drop(columns=["pipeline"]))
            


In [ ]:
best_row = results_df.iloc[0]
final_model = best_row["pipeline"]
final_model_name = best_row["model"]
y_pred = final_model.predict(X_test)

print(f"Selected best model: {final_model_name}")
print("Best params:")
print(best_row["best_params"])
print()
print(classification_report(y_test, y_pred, target_names=class_names, zero_division=0))

logger.report_scalar("final/test_f1_weighted", final_model_name, float(best_row["test_f1_weighted"]), 0)
logger.report_scalar("final/test_precision_weighted", final_model_name, float(best_row["test_precision_weighted"]), 0)
logger.report_scalar("final/test_recall_weighted", final_model_name, float(best_row["test_recall_weighted"]), 0)
logger.report_scalar("final/test_accuracy", final_model_name, float(best_row["test_accuracy"]), 0)

task.connect({
    "selected_model": final_model_name,
    "selected_model_params": best_row["best_params"],
})
            


In [ ]:
ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_pred,
    display_labels=class_names,
    cmap="Blues",
    xticks_rotation=20,
)
plt.title(f"{final_model_name} Confusion Matrix")
plt.tight_layout()
plt.show()
            


In [ ]:
output_path = Path("..") / "artifact" / "cars_hyundai_notebook_best_model.pkl"
output_path.parent.mkdir(parents=True, exist_ok=True)

bundle = {
    "model_name": final_model_name,
    "pipeline": final_model,
    "label_encoder": label_encoder,
    "target_column": TARGET_COLUMN,
    "feature_columns": NUMERIC_FEATURES,
    "results": results_df.drop(columns=["pipeline"]).to_dict(orient="records"),
}

with open(output_path, "wb") as file_obj:
    pickle.dump(bundle, file_obj)

print(f"Saved best model bundle to: {output_path.resolve()}")
task.upload_artifact("cars_hyundai_best_model", artifact_object=str(output_path.resolve()))
            


## Outcome

- The target is now `maintenance_type`, not `anomaly_indication`.
- `anomaly_indication` is treated as an input feature.
- Random Forest and SVM both go through Optuna HPO.
- ClearML records the HPO curves, comparison table, final metrics, and saved artifact.
            
